# Import the Dataframe

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import seaborn as sns

from matplotlib.colors import LogNorm, SymLogNorm
import matplotlib.dates as mdates
from scipy.stats import linregress, ttest_ind
from shapely.geometry import Point
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor

In [ ]:
df = pd.read_excel("cleaned_data.xlsx")
df = df.dropna(subset=["alt"])
df

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
geo_rad = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

COLOR = "managua"

### Create a column that separates code by latitude region

In [ ]:
geo_rad["region"] = geo_rad["lat"].case_when(
    caselist=[
        (geo_rad["lat"] >= 66.5, "Arctic"),
        (geo_rad["lat"] >= 23.5, "North Temperate"),
        (geo_rad["lat"] >= -23.5, "Equator"),
        (geo_rad["lat"] >= -66.5, "South Temperate"),
        (geo_rad["lat"] >= -200, "Antarctic")
    ]
)

# Define the explicit order you want to see on the plot
desired_order = ['Arctic', 'North Temperate', 'Equator', 'South Temperate', "Antarctic"]

# Convert the column into an ordered categorical category
geo_rad['category'] = pd.Categorical(geo_rad['region'], categories=desired_order, ordered=True)

# Altitude, Latitude, and Radiation Exploration

## Altitude vs Latitude

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

geo_rad.boxplot(column="alt", 
                by="category", 
                ax=ax,
                patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6),
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker="o", markersize=2, alpha=0.3))

plt.suptitle("")
ax.set_title("Altitudes by Latitude Zones")
ax.set_ylabel("Altitude (km)")
ax.set_xlabel("Latitude Zone")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad.plot(ax=ax,
            markersize=10,
            alpha=0.5,
            column="alt", 
            cmap=COLOR,
            legend=True,
            legend_kwds = {"label": "Altitude (km)"}
            )

ax.set_aspect("equal", adjustable="box")
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")

plt.title("Altitude of the satellite")
plt.show()

I was expecting the altitudes to be highest near the poles, but it appears it's actually second lowest in the Arctic and highest in the Antarctic. Similarly, I was expecting altitude to be lowest at the poles, but it appears it was close to the same as Arctic. I'm not an expert on satellite flight patterns, but this makes me think that the satellite's orbit is eliptical, and it is gaining its speed by dipping in in the Northern Temperate zone. I suspect the satellite has enough speed that it gains some altitude over the Antarctic before crossing over the North Pole.

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

time_num = mdates.date2num(df["timestamp"])

fig, ax = plt.subplots(figsize=(18, 9))

sc = ax.scatter(
    df["alt"],
    df["lat"],
    c=time_num,
    cmap="viridis",
    s=5
)

cbar = plt.colorbar(sc)
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
cbar.set_label("Timestamp", fontsize="x-large")

ax.set_ylabel("Latitude", fontsize="x-large")
ax.set_xlabel("Altitude", fontsize="x-large")
ax.set_title("Altitude vs Latitude and Timestamp", fontsize="xx-large")

plt.show()

This visual shows us that every latitude has roughly a 20km altitude window in which it collected data for that latitude/altitude pairing. Another interesting thing to note is that the satellite appears to have been losing altitude throughout the duration of its time in orbit as the higher altitude datapoints for each latitude is a darker shade indicating an early timestamp, while the lower altitude datapoints for each latitude is a lighter shade indicating an later timestamp.

### Adding on In Shadow

Since the satellite is always moving south when in sunlight and always moving north in shadow, this might allow us to see something interesting about the orbit of the satellite.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3), constrained_layout=True)

for shadow in [0, 1]:
    subset = geo_rad[geo_rad["in_shadow"] == shadow]

    world.plot(ax=axes[shadow], color="lightgrey")
    subset.plot(ax=axes[shadow],
                markersize=10,
                alpha=0.5,
                column="alt", 
                cmap=COLOR,
                legend=True,
                legend_kwds = {"label": "Altitude (km)"}
                )

    axes[shadow].set_title(f"Altitude of the satellite when in_shadow == {shadow}")
    axes[shadow].set_xlabel("Longitude")
    axes[shadow].set_ylabel("Latitude")

plt.suptitle("Altitude split by in_shadow", fontsize="xx-large")
plt.show()

I think I might have been right. The altitude while going north dips down and then goes back up when nearing the poles, and then it stays fairly low most of the time it's heading south. I think the satellite might be sort of "falling" as it dips down, gaining speed, and this speed helps it rise back up and continue its orbit.

## Bringing in Radiation

### Viewing Radiation by Bins

In [ ]:
geo_rad["alt_bin"] = geo_rad["alt"].case_when(
    caselist=[
        (geo_rad["alt"] <= 480, "468 - 480"),
        (geo_rad["alt"] <= 492, "480 - 492"),
        (geo_rad["alt"] <= 504, "492 - 504"),
        (geo_rad["alt"] <= 516, "504 - 516")
    ]
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)

for i, rad in enumerate(["xray0_ps", "proton0_ps", "electron0_ps"]):
    geo_rad.boxplot(column=rad, 
                    by="alt_bin", 
                    ax=axes[i],
                    patch_artist=True,
                    boxprops=dict(facecolor="steelblue", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2),
                    flierprops=dict(marker="o", markersize=2, alpha=0.3))
    axes[i].set_yscale("log")
    axes[i].set_title(f"{rad} by altitude")
    axes[i].set_ylabel(f"{rad} (Log)")
    axes[i].set_xlabel("Altitude (km)")

plt.suptitle("Radiation Binned by Altitude", fontsize="xx-large")  # removes the automatic "Boxplot grouped by kp" title
plt.show()

**Analysis of boxplot**
- It appears that X-ray radiation becomes lower the higher you go in the atmosphere
    - This brings up something interesting: The times where the satellite is at its highest is when the satellite is over the Antarctic and when it's near the poles in shadow, the times where we've observed the lowest X-ray values. 
    - This brings up an important question: is altitude affecting our X-ray counts, is it overlapping with latitude and sunlight by pure chance, or do all three play a role?
- Protons don't seem to vary quite as much with altitude
- Electrons appear higher the higher in the atmosphere you go

In [ ]:
geo_rad['alt_1km'] = geo_rad['alt'].round(0)  # or use a finer pd.cut if data is dense enough

summary = geo_rad.groupby('alt_1km')['xray0_ps'].agg(['median', 'mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(summary['alt_1km'], summary['median'], marker='o', markersize=3)
ax.set_yscale('log')
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('xray0_ps median (log)')
ax.set_title('Median xray0_ps vs altitude (fine resolution)')
ax.grid(True, alpha=0.3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(summary['alt_1km'], summary['count'], marker='o', markersize=3)
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Bin size')
ax.set_title('Median xray0_ps vs bin size (fine resolution)')
ax.grid(True, alpha=0.3)

plt.show()

In [ ]:
df_sorted = geo_rad.sort_values('alt')
df_sorted['xray_rolling_median'] = df_sorted['xray0_ps'].rolling(window=200, center=True).median()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_sorted['alt'], df_sorted['xray_rolling_median'])
ax.set_yscale('log')
ax.set_xlabel('Altitude (km)')
ax.set_ylabel('Rolling median xray0_ps (log)')

plt.show()

**Analysis of finer bins**

This shows that the dropoff isn't exponential either. It oscillates some. This makes sense given the orbital nature of our data. Because analyzing the data this way involved sorting it out of chronological order, it might be better to look at it in time order.

### Scatterplot Pairings

In [ ]:
# - Linear from 0 to 5000
# - Logarithmic above 5000
norm = SymLogNorm(
    linthresh=5000,
    linscale=1,
    vmin=df["xray0_ps"].min(),
    vmax=df["xray0_ps"].max()
)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(12, 12),
    sharex=True,
    constrained_layout=True
)

sc = ax1.scatter(
    df["alt"],
    df["lat"],
    c=df["xray0_ps"],
    cmap="viridis",
    norm=norm,
    s=2
)

ax1.set_ylabel("Latitude")
ax1.set_title("Altitude, Latitude, and X-Ray Radiation")

ax2.scatter(
    df["alt"],
    df["xray0_ps"],
    c=df["xray0_ps"],
    cmap="viridis",
    norm=norm,
    s=2
)

ax2.set_xlabel("Altitude")
ax2.set_ylabel("X-Ray Radiation")

cbar = fig.colorbar(sc, ax=[ax1, ax2])
cbar.set_label("X-Ray Radiation")

plt.show()

We see a large spike in x-ray radiation at an altitude around 480km. However, as we inspect where these higher values occured in the top scatterplot, we can see that they come in the arctic circle in the later parts of our dataset (March/April). This remains consistent with our preliminary findings. 

This then begs the question is an altitude of 480km significant to x-ray radiation or is it rehighlighting the hotspot of x-ray radiation found in the arctic circle in the early spring months? 

In [ ]:
# - Linear from 0 to 5000
# - Logarithmic above 5000
norm = SymLogNorm(
    linthresh=5000,
    linscale=1,
    vmin=df["proton0_ps"].min(),
    vmax=df["proton0_ps"].max()
)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(12, 12),
    sharex=True,
    constrained_layout=True
)

sc = ax1.scatter(
    df["alt"],
    df["lat"],
    c=df["proton0_ps"],
    cmap="viridis",
    norm=norm,
    s=2
)

ax1.set_ylabel("Latitude")
ax1.set_title("Altitude, Latitude, and Proton Radiation")

ax2.scatter(
    df["alt"],
    df["proton0_ps"],
    c=df["proton0_ps"],
    cmap="viridis",
    norm=norm,
    s=2
)

ax2.set_xlabel("Altitude")
ax2.set_ylabel("Proton Radiation")

cbar = fig.colorbar(sc, ax=[ax1, ax2])
cbar.set_label("Proton Radiation")

plt.show()

In [ ]:
# - Linear from 0 to 1000
# - Logarithmic above 1000
norm = SymLogNorm(
    linthresh=1000,
    linscale=1,
    vmin=df["electron0_ps"].min(),
    vmax=df["electron0_ps"].max()
)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(12, 12),
    sharex=True,
    constrained_layout=True
)

sc = ax1.scatter(
    df["alt"],
    df["lat"],
    c=df["electron0_ps"],
    cmap="viridis",
    norm=norm,
    s=2
)

ax1.set_ylabel("Latitude")
ax1.set_title("Altitude, Latitude, and Electron Radiation")

ax2.scatter(
    df["alt"],
    df["electron0_ps"],
    c=df["electron0_ps"],
    cmap="viridis",
    norm=norm,
    s=2
)

ax2.set_xlabel("Altitude")
ax2.set_ylabel("Electron Radiation")

cbar = fig.colorbar(sc, ax=[ax1, ax2])
cbar.set_label("Electron Radiation")

plt.show()

### Timestamp Analysis

In [ ]:
geo_rad_time_sorted = geo_rad.sort_values('timestamp')

plot_config = [
    ('xray0_ps',  'median', 'xray0_ps',      'Timestamp vs xray_ps',   'log'),
    ('alt',       'median', 'alt (km)',      'Timestamp vs altitude',  'linear'),
    ('lat',       'median', 'lat',           'Timestamp vs latitude',  'linear'),
    ('in_shadow', 'mean',   'in_shadow (frac)', 'Timestamp vs in_shadow', 'linear'),
]

fig, axes = plt.subplots(len(plot_config), 1, figsize=(12, 10), sharex=True)

for ax, (col, agg, ylabel, title, yscale) in zip(axes, plot_config):
    rolled = geo_rad_time_sorted[col].rolling(200, center=True).agg(agg)
    ax.plot(geo_rad_time_sorted['timestamp'], rolled)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if yscale == 'log':
        ax.set_yscale('log')
    ax.tick_params(labelbottom=True)

axes[-1].set_xlabel('Timestamp')
plt.tight_layout()

**Analysis of Timestamp vs X-Ray Radiaiton, Altitude, Latitude, and In Shadow**

A few important things to note:
- The increase in x-ray radiation with time stays consistent with what we've already observed, meaning the order of the data is intact
- Barring the one spike, altitude appears to be slowly decreasing as time goes on
- At the same time as the altitude spike, latitude dips dramatically and goes much further than we've seen before
- Timestamp and latitude mimic each other's movements surprisngly well until about March

That spike in altitude and dip in latitude tell an interesting story. It almost seems like the satellite deviated from its normal course for some reason. Even more interestingly, X-ray readings don't drop dramatically like you might expect when altitude goes up and latitude goes down. This might be worth looking into in greater detail.

### Tests

In [ ]:
def lin_reg_ttest(x,y,d):
    for i in x:
        for j in y:
            result = linregress(d[i],d[j])
            slope = result.slope
            intercept = result.intercept
            r_squared = result.rvalue**2
            p_value = result.pvalue
            t_stat = result.slope / result.stderr
            
            y_fit = slope * d[i] + intercept
            
            plt.figure(figsize=(15, 8))
            plt.scatter(d[i], d[j], s=5, alpha=0.5)
            plt.plot(d[i], y_fit, color="red")
            
            plt.xlabel(i)
            plt.ylabel(j)
            plt.title(i + ' vs ' + j)
            
            plt.show()
            
            print(f"Slope: {slope:.6f}")
            print(f"Intercept: {intercept:.6f}")
            print(f"R²: {r_squared:.4f}")
            print(f"t-statistic: {t_stat:.4f}")
            print(f"p-value: {p_value:.6g}")

            decision = "Reject Null" if p_value < 0.05 else "Fail to Reject"
            print(f"Decision: {decision}")

In [ ]:
xlabs1 = ['alt']
ylabs1 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs1,ylabs1,df)

**Linear Regression t-Tests**

The results for the linear regression t-test between altitude and x-ray radiation is the most significant result we have gotten on a linear regression test thus far. There is a statistically significant negative slope as well as an $R^2$ of 0.1225, which is the highest $R^2$ value we have had. 

As for the t-tests of proton and electron radiation versus altitude, we see similar results as always with a statistically significant slope and a very low $R^2$ value.

In [ ]:
corr = df[["lat", "alt", "in_shadow", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)
ax.xaxis.tick_top()
plt.show()

In [ ]:
X = geo_rad[['lat', 'alt', 'in_shadow']].dropna()
y = geo_rad.dropna()["xray0_ps"]

mi = mutual_info_regression(X, y)
print(f"Mutual info regression:\n{pd.Series(mi, index=X.columns).sort_values(ascending=False)}\n\n")

rf = RandomForestRegressor(n_estimators=200, random_state=0)
rf.fit(X, y)
print(f"Random Forest Regression:\n{pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)}")

**Analysis of correlation comparison**

Both mutual info and random forest regression agree that altitude is the driving force for x-ray radiation, while the correlation matrix shows the satellite being in the Earth's shadow has the highest correlation. This indicates a more linear relationship with the Earth's shadow, and a noticable but nonlinear relationship with altitude. This makes sense looking at the boxplot of radiations. X-ray radiation did not decrease linearly as altitude increases.

It's worth noting that all three variables have a moderate strength correlation with x-ray radiation. 

# Tempature

In [ ]:
df2 = pd.read_excel(r"C:\Users\n_jac\RadStar\Book1.xlsx")
df2

In [ ]:
df2 = df2.dropna(subset=["S4 Health + Safety v2"]).copy()
df2

In [ ]:
df2 = df2.iloc[1:].reset_index(drop=True)

df2["Unnamed: 0"] = pd.to_datetime(df2["Unnamed: 0"])

df2copy = df2[
    (df2["Unnamed: 0"] >= "2025-01-02") &
    (df2["Unnamed: 0"] <= "2025-04-9")
].copy()

df2copy

The original tempature dataset provided to us from NearSpace Launch contained a total of $45,243$ datapoints. However, it contained a huge number of nulls with $97.58% (44,147)$ of the $45,243$ rows being null. Then when we restrict these remaining $1,096$ non-null datapoints to our timeframe of January 2nd to April 9th, we are left with only $84$ datapoints to work with. This is $0.19%$ of our original dataset and trying to concatinate $84$ tempature datapoints onto our radiation dataset of over $8,000$ datapoints is not going to be very accurate or beneficial. 

Then when we look at back at the radios that the tempatures came from, we can see here that radioIDs `10006` and `10162` are both entirely nulls. Then when we look at the 84 non-null temperature values in the time window of our data. 81 of them were ID `10009`, and 3 of them were ID `10012`, but all three of ID `10012` had a value of ID `10009` take place before another RadStar packet took place. This is how the above graph occured. All of our temerature data (with the way I backfilled the data) came from the radio with ID `10009`.